In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

VAL_PATH = r"C:\EOL_EDA\clean_merged_valeol.csv"

df_val = pd.read_csv(VAL_PATH, low_memory=False)
print(f"Raw shape: {df_val.shape}")
print(df_val['outcome_from_filename'].value_counts())

In [ ]:
df_val.columns = (
    df_val.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^a-z0-9_]', '', regex=True)
)

df_val = df_val[~df_val['no'].astype(str).str.lower().isin(['no', 'nan'])]
df_val = df_val[~df_val['step_name_1'].astype(str).str.lower().str.contains('step name', na=False)]
df_val.reset_index(drop=True, inplace=True)

df_val['unit_id']               = df_val['source_file'].astype(str).str.strip()
df_val['result']                = df_val['result'].astype(str).str.strip().str.upper()
df_val['outcome_from_filename'] = df_val['outcome_from_filename'].astype(str).str.strip().str.upper()
df_val['result_binary']         = df_val['result'].map({'PASS': 1, 'FAIL': 0, 'ABORT': 0})
df_val['unit_pass']             = df_val['outcome_from_filename'].map({'PASS': 1, 'FAIL': 0, 'ABORT': 0})
df_val['step_time']             = pd.to_numeric(df_val['step_time'], errors='coerce').fillna(0.0)
df_val['step_name_1']           = df_val['step_name_1'].astype(str).str.strip()
df_val['step_name_2']           = df_val['step_name_2'].astype(str).str.strip()
df_val['test_id']               = df_val['step_name_1'] + " / " + df_val['step_name_2']

print(f"Unique units:      {df_val['unit_id'].nunique()}")
print(f"Unique test steps: {df_val['test_id'].nunique()}")

In [ ]:
unit_summary_val = df_val.groupby('unit_id').agg(
    overall_outcome    = ('unit_pass',     'first'),
    total_test_time    = ('step_time',     'sum'),
    num_steps_executed = ('test_id',       'count'),
    num_steps_failed   = ('result_binary', lambda x: (x == 0).sum())
).reset_index()

all_failing_val  = unit_summary_val[unit_summary_val['overall_outcome'] == 0]['unit_id'].tolist()
genuine_fail_val = []
abort_only_val   = []

for unit_id in all_failing_val:
    unit_steps    = df_val[df_val['unit_id'] == unit_id]
    has_step_fail = (unit_steps['result'] == 'FAIL').any()
    if has_step_fail:
        genuine_fail_val.append(unit_id)
    else:
        abort_only_val.append(unit_id)

failing_units_val = genuine_fail_val
N_fail_val        = len(failing_units_val)
n_total           = len(unit_summary_val)

print(f"{'='*55}")
print("EOL VALIDATION DATASET SUMMARY")
print(f"{'='*55}")
print(f"Total units:          {n_total}")
print(f"Genuine FAIL units:   {N_fail_val}")
print(f"ABORT-only excluded:  {len(abort_only_val)}")
print(f"Defect rate:          {N_fail_val/n_total*100:.1f}%")

In [ ]:
outcome_matrix_val = df_val.pivot_table(
    index='unit_id', columns='test_id',
    values='result_binary', aggfunc='first'
)

cost_vector = pd.read_csv("eol_cost_vector.csv", index_col=0, header=0)
cost_vector.columns = ['mean_cost_s']
cred_df    = pd.read_csv("eol_rq2_cred.csv")
C_red_list = cred_df['test_id'].tolist()

C_full_cost = cost_vector['mean_cost_s'].sum()
C_red_cost  = sum(
    cost_vector.loc[s, 'mean_cost_s']
    for s in C_red_list if s in cost_vector.index
)

print(f"Outcome matrix:   {outcome_matrix_val.shape}")
print(f"Cred steps:       {len(C_red_list)}")
print(f"C_full_cost:      {C_full_cost:.2f} s")
print(f"C_red_cost:       {C_red_cost:.2f} s")

In [ ]:
PASS_RATE_WINDOW    = 100   # updated from 50
PASS_RATE_THRESHOLD = 0.93
ESCAPE_PENALTY      = 10
THRESHOLD       = 85

seen, unit_stream_val = set(), []
for u in df_val['unit_id'].tolist():
    if u not in seen:
        unit_stream_val.append(u)
        seen.add(u)

unit_outcome_map_val = unit_summary_val.set_index('unit_id')['overall_outcome'].to_dict()
unit_pass_seq_val    = np.array([unit_outcome_map_val.get(u, 1) for u in unit_stream_val])

rolling_pass_rate_val = np.zeros(len(unit_stream_val))
for i in range(len(unit_stream_val)):
    window_start = max(0, i - PASS_RATE_WINDOW + 1)
    rolling_pass_rate_val[i] = unit_pass_seq_val[window_start:i+1].mean()

print(f"Validation stream:  {len(unit_stream_val)} units")
print(f"Pass rate — mean:   {rolling_pass_rate_val.mean():.4f}")
print(f"Pass rate — min:    {rolling_pass_rate_val.min():.4f}")
print(f"% unstable:         {(rolling_pass_rate_val < PASS_RATE_THRESHOLD).mean()*100:.1f}%")
print(f"Pass rate at idx=2772: {rolling_pass_rate_val[2772]:.4f}")

In [ ]:
def unit_is_defective_val(uid):
    return uid in failing_units_val

def cred_detects_val(uid):
    if uid not in outcome_matrix_val.index:
        return True
    row = outcome_matrix_val.loc[uid]
    return any(s in row.index and row[s] == 0 for s in C_red_list)

def run_ts_val(boost):
    alpha_ts, beta_ts = [5.0, 1.0], [1.0, 1.0]
    choices, costs_log, escapes_log = [], [], []
    escaped = 0
    np.random.seed(42)

    for i, uid in enumerate(unit_stream_val):
        pass_rate  = rolling_pass_rate_val[i]
        theta_full = np.random.beta(alpha_ts[0], beta_ts[0])
        theta_red  = np.random.beta(alpha_ts[1], beta_ts[1])

        if pass_rate < PASS_RATE_THRESHOLD:
            theta_full += (PASS_RATE_THRESHOLD - pass_rate) * boost

        chosen = 1 if theta_red > theta_full else 0
        choices.append(chosen)
        costs_log.append(C_full_cost if chosen == 0 else C_red_cost)

        is_def = unit_is_defective_val(uid)
        if chosen == 1 and is_def and not cred_detects_val(uid):
            reward, escaped = -ESCAPE_PENALTY, escaped + 1
        elif chosen == 1 and is_def:
            reward = 0.5
        elif chosen == 1:
            reward = 1.0
        else:
            reward = 0.0

        escapes_log.append(escaped)
        r = max(0.0, min(1.0, reward))
        if r >= 0.5: alpha_ts[chosen] += 1
        else:        beta_ts[chosen]  += 1

    pct_cred   = choices.count(1) / len(choices) * 100
    saving_pct = (C_full_cost - np.mean(costs_log)) / C_full_cost * 100
    return {
        'choices':     choices,
        'costs':       costs_log,
        'escapes_log': escapes_log,
        'escaped':     escaped,
        'pct_cred':    round(pct_cred, 1),
        'saving_pct':  round(saving_pct, 2),
        'escape_risk': round(escaped / max(N_fail_val, 1), 6),
        'quality_ok':  'YES' if escaped/max(N_fail_val,1)*1e6 <= THRESHOLD else 'NO'
    }

print("Helper functions defined")

In [ ]:
result_b10  = run_ts_val(10)
result_b50  = run_ts_val(50)
result_b100 = run_ts_val(100)

print(f"β=10  — Cred: {result_b10['pct_cred']}%, "
      f"Saving: {result_b10['saving_pct']}%, "
      f"Escaped: {result_b10['escaped']}, "
      f"Quality: {result_b10['quality_ok']}")
print(f"β=50  — Cred: {result_b50['pct_cred']}%, "
      f"Saving: {result_b50['saving_pct']}%, "
      f"Escaped: {result_b50['escaped']}, "
      f"Quality: {result_b50['quality_ok']}")
print(f"β=100 — Cred: {result_b100['pct_cred']}%, "
      f"Saving: {result_b100['saving_pct']}%, "
      f"Escaped: {result_b100['escaped']}, "
      f"Quality: {result_b100['quality_ok']}")

In [ ]:
escaped_static     = sum(1 for u in failing_units_val if not cred_detects_val(u))
saving_static      = (C_full_cost - C_red_cost) / C_full_cost * 100
escape_risk_static = escaped_static / max(N_fail_val, 1)

rows = [
    {'Period':'Validation', 'Algorithm':'Baseline (Cfull)',  'Beta':'---',
     'Cred (%)':0.0,   'Saving (%)':0.0,
     'Escaped':0, 'R_hat':0.0, 'Quality':'YES'},
    {'Period':'Validation', 'Algorithm':'Static Cred (RQ1)', 'Beta':'---',
     'Cred (%)':100.0, 'Saving (%)':round(saving_static,2),
     'Escaped':escaped_static,
     'R_hat':round(escape_risk_static,6),
     'Quality':'YES' if escape_risk_static*1e6 <= THRESHOLD else 'NO'},
    {'Period':'Validation', 'Algorithm':'Thompson Sampling', 'Beta':'10',
     'Cred (%)':result_b10['pct_cred'],
     'Saving (%)':result_b10['saving_pct'],
     'Escaped':result_b10['escaped'],
     'R_hat':result_b10['escape_risk'],
     'Quality':result_b10['quality_ok']},
    {'Period':'Validation', 'Algorithm':'Thompson Sampling', 'Beta':'50',
     'Cred (%)':result_b50['pct_cred'],
     'Saving (%)':result_b50['saving_pct'],
     'Escaped':result_b50['escaped'],
     'R_hat':result_b50['escape_risk'],
     'Quality':result_b50['quality_ok']},
    {'Period':'Validation', 'Algorithm':'Thompson Sampling', 'Beta':'100',
     'Cred (%)':result_b100['pct_cred'],
     'Saving (%)':result_b100['saving_pct'],
     'Escaped':result_b100['escaped'],
     'R_hat':result_b100['escape_risk'],
     'Quality':result_b100['quality_ok']},
]

val_df = pd.DataFrame(rows)
val_df.to_csv("eol_val_final_results.csv", index=False)
print("\nEOL VALIDATION RESULTS (window=100, τ=0.93)")
print("="*75)
print(val_df.to_string(index=False))

In [ ]:
exec_index_val = np.arange(len(unit_stream_val))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(exec_index_val, rolling_pass_rate_val,
        color='black', linewidth=1.5,
        label=f'Rolling pass rate (w={PASS_RATE_WINDOW})')
ax.axhline(y=PASS_RATE_THRESHOLD, color='red',
           linestyle='--', linewidth=1.5,
           label=f'Stability threshold ({PASS_RATE_THRESHOLD})')
ax.set_xlabel('Execution index (EOL events in time order)',
              fontsize=14, fontweight='bold')
ax.set_ylabel(f'Rolling pass rate (w={PASS_RATE_WINDOW})',
              fontsize=14, fontweight='bold')
ax.legend(fontsize=10)

ax.set_xlim(0, 4500)
ax.set_ylim(0, 1)
ax.yaxis.set_major_locator(plt.MultipleLocator(0.1))
ax.xaxis.set_major_locator(plt.MultipleLocator(600))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3)
ax.tick_params(axis='both', labelsize=11)

plt.tight_layout()
plt.savefig('eol_val_rolling_passrate.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eol_val_rolling_passrate.png")

In [ ]:
window           = 300
choices_arr_val  = np.array(result_b100['choices'])
rolling_cred_val = pd.Series(choices_arr_val).rolling(
    window=window, min_periods=1
).mean().values

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(exec_index_val, rolling_cred_val,
        color='black', linewidth=2,
        label=f'Fraction selecting Cred (w={window}, β=100)')
ax.axhline(y=0.5, color='red', linestyle='--',
           linewidth=1.5, label='50% threshold')
ax.set_xlabel('Execution index (EOL events in time order)',
              fontsize=14, fontweight='bold')
ax.set_ylabel('Fraction selecting Cred',
              fontsize=14, fontweight='bold')
ax.legend(fontsize=10)

ax.set_xlim(0, 4200)
ax.set_ylim(0, 1)
ax.yaxis.set_major_locator(plt.MultipleLocator(0.1))
ax.xaxis.set_major_locator(plt.MultipleLocator(600))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3)
ax.tick_params(axis='both', labelsize=11)

plt.tight_layout()
plt.savefig('eol_val_arm_selection.png', 
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eol_val_arm_selection.png")

In [ ]:
# Regenerate sensitivity with correct data including beta=100 zero escape
boost_vals = [1, 5, 10, 20, 30, 50, 75, 100]
sens_rows  = []
for b in boost_vals:
    r = run_ts_val(b)
    sens_rows.append({
        'Boost factor': b,
        'Saving (%)':   r['saving_pct'],
        'Escaped':      r['escaped'],
        'Escape risk':  r['escape_risk']
    })

sens_df = pd.DataFrame(sens_rows)
print(sens_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
boost_labels = [str(b) for b in sens_df['Boost factor']]

# Left — cost saving
axes[0].bar(boost_labels, sens_df['Saving (%)'],
            color='steelblue', alpha=0.8,
            edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Instability sensitivity parameter (β)',
                   fontsize=13, fontweight='bold')
axes[0].set_ylabel('Cost Saving (%)', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].grid(True, alpha=0.3, axis='y')
for j, v in enumerate(sens_df['Saving (%)']):
    axes[0].text(j, v + 0.5, f'{v:.2f}%',
                 ha='center', fontsize=9, fontweight='bold')

# Right — escape risk with correct colors
bar_colors = [
    'steelblue' if r > 0 else 'green'
    for r in sens_df['Escape risk']
]
bars = axes[1].bar(boost_labels, sens_df['Escape risk'],
                   color=bar_colors, alpha=0.8,
                   edgecolor='black', linewidth=0.5)
axes[1].axhline(y=THRESHOLD/1e6, color='red',
                linestyle='--', linewidth=1.5,
                label=f'Quality threshold ({THRESHOLD})')
axes[1].set_xlabel('Instability sensitivity parameter (β)',
                   fontsize=13, fontweight='bold')
axes[1].set_ylabel('Escape risk R̂(π)', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 0.015)  # force scale to show zero clearly
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].legend(fontsize=10)

# Label each bar with value
for j, v in enumerate(sens_df['Escape risk']):
    label = '0.000' if v == 0 else f'{v:.4f}'
    axes[1].text(j, v + 0.0003, label,
                 ha='center', fontsize=9, fontweight='bold',
                 color='green' if v == 0 else 'black')

# Add green patch to legend for zero escape
from matplotlib.patches import Patch
handles, labels = axes[1].get_legend_handles_labels()
handles.append(Patch(color='green', alpha=0.8, label='Zero escapes'))
axes[1].legend(handles=handles, fontsize=10)

plt.tight_layout()
plt.savefig('eol_val_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eol_val_sensitivity.png")